<a href="https://colab.research.google.com/github/tingtingtingtin/SEARCHME/blob/sazen%2Fml-initial-setup/searchme_spark_embedding_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- STEP 1: INSTALLATION & AUTH ---
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.0 sentence_transformers -q

import os
import pyspark

# Find the real Spark JARs directory
spark_jars_path = os.path.join(os.path.dirname(pyspark.__file__), "jars")

# 1. Download BigQuery Connector
!wget https://repo1.maven.org/maven2/com/google/cloud/spark/spark-bigquery-with-dependencies_2.12/0.34.0/spark-bigquery-with-dependencies_2.12-0.34.0.jar -O {spark_jars_path}/spark-bigquery.jar

# 2. Download GCS Connector
!wget https://repo1.maven.org/maven2/com/google/cloud/bigdataoss/gcs-connector/hadoop3-2.2.14/gcs-connector-hadoop3-2.2.14-shaded.jar -O {spark_jars_path}/gcs-connector.jar

from google.colab import auth
auth.authenticate_user()
!gcloud auth application-default login --no-launch-browser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
--2026-03-07 23:18:57--  https://repo1.maven.org/maven2/com/google/cloud/spark/spark-bigquery-with-dependencies_2.12/0.34.0/spark-bigquery-with-dependencies_2.12-0.34.0.jar
Resolving repo1.maven.org (repo1.maven.org)... 104.18.18.12, 104.18.19.12, 2606:4700::6812:130c, ...
Connecting to repo1.maven.org (repo1.maven.org)|104.18.18.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 39453989 (38M) [application/java-archive]
Saving to: ‘/usr/local/lib/python3.12/dist-packages/pyspar

In [22]:
# --- STEP 2: STABLE SPARK INITIALIZATION ---
from pyspark.sql import SparkSession
import os

PROJECT_ID = 'search-me-cs226'
STAGING_BUCKET = 'search-me-cs226-staging'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

if 'spark' in globals():
    try: spark.stop()
    except: pass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SearchMe_Production") \
    .config("spark.driver.memory", "8g") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.conf.set("temporaryGcsBucket", STAGING_BUCKET)

# Verify BOTH drivers
try:
    spark._jvm.java.lang.Class.forName("com.google.cloud.spark.bigquery.DefaultSource")
    spark._jvm.java.lang.Class.forName("com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    print("🚀 SUCCESS: BigQuery and GCS drivers are verified and READY!")
except Exception as e:
    print(f"❌ Verification failed. Spark is still not seeing the JARs: {e}")

🚀 SUCCESS: BigQuery and GCS drivers are verified and READY!


In [23]:
# --- STEP 3: DEFINE LOGIC (UDFs) ---
import pandas as pd
import torch
import time
import re
from pyspark.sql.functions import col, pandas_udf, posexplode, expr, udf
from pyspark.sql.types import ArrayType, StringType, FloatType, DoubleType
from sentence_transformers import SentenceTransformer

# Use a cache to ensure the model is only loaded once per Spark executor
_model_cache = {}

def get_model():
    if "all-minilm" not in _model_cache:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _model_cache["all-minilm"] = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    return _model_cache["all-minilm"]

def clean_markdown(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove image markdown
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    # Remove hyperlinks but keep link text
    text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text)
    # Remove badge/shield URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove code blocks
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'`[^`]*`', '', text)
    # Remove markdown headers
    text = re.sub(r'#{1,6}\s*', '', text)
    # Remove markdown bold/italic
    text = re.sub(r'\*{1,3}(.*?)\*{1,3}', r'\1', text)
    text = re.sub(r'_{1,3}(.*?)_{1,3}', r'\1', text)
    # Remove horizontal rules
    text = re.sub(r'^\s*[-*_]{3,}\s*$', '', text, flags=re.MULTILINE)
    # Remove bullet points and numbered lists markers
    text = re.sub(r'^\s*[-*+]\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\d+\.\s+', '', text, flags=re.MULTILINE)
    # Collapse excessive whitespace and newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

clean_markdown_udf = udf(clean_markdown, StringType())

print("✅ Markdown cleaning UDF defined.")

@pandas_udf(ArrayType(StringType()))
def chunk_text_udf(texts: pd.Series) -> pd.Series:
    model = get_model()
    max_length, overlap = 256, 30
    stride = max_length - overlap
    all_chunks = []

    for text in texts:
        if not isinstance(text, str) or not text.strip():
            all_chunks.append([])
            continue
        tokens = model.tokenizer(text, add_special_tokens=False, truncation=False)['input_ids']
        chunks = [model.tokenizer.decode(tokens[i : i + max_length])
                  for i in range(0, len(tokens), stride)]
        all_chunks.append(chunks)
    return pd.Series(all_chunks)

@pandas_udf(ArrayType(DoubleType()))
def generate_embeddings_udf(texts: pd.Series) -> pd.Series:
    model = get_model()
    if texts.isnull().all():
        return pd.Series([[]] * len(texts))

    embeddings = model.encode(
        texts.fillna("").tolist(),
        batch_size=64,
        show_progress_bar=False
    )
    return pd.Series([v.astype("float64").tolist() for v in embeddings])

print("✅ Logic and UDFs updated with robust error handling.")

✅ Markdown cleaning UDF defined.
✅ Logic and UDFs updated with robust error handling.


In [24]:
from pyspark.sql.functions import size, trim, col, posexplode, expr
from google.cloud import bigquery
import time

# --- STEP 4: SECURE EXECUTE (V2) ---
SOURCE_TABLE = f"{PROJECT_ID}.searchme_dataset.cleaned_readme_subset"
DEST_TABLE = f"{PROJECT_ID}.searchme_dataset.embeddings_spark_224_clean"
GCS_PARQUET_PATH = f"gs://{STAGING_BUCKET}/embeddings_output/spark_v2"

print(f"ፁ Processing from {SOURCE_TABLE} to {DEST_TABLE}...")

df = spark.read.format("bigquery").option("table", SOURCE_TABLE).load()

# 1. Chunk, explode, embed and filter
final_df = df.withColumn("readme_text", clean_markdown_udf(col("readme_text"))) \
    .withColumn("chunks", chunk_text_udf(col("readme_text"))) \
    .select("repo_name", posexplode(col("chunks")).alias("chunk_idx", "chunk_text")) \
    .withColumn("chunk_id", expr("concat(repo_name, '_chunk_', cast(chunk_idx as string))")) \
    .filter(trim(col("chunk_text")) != "") \
    .withColumn("embedding", generate_embeddings_udf(col("chunk_text"))) \
    .filter(size(col("embedding")) == 384)

# 2. Write to GCS as Parquet
print("⚙ Writing to GCS as Parquet...")
final_df.write.mode("overwrite").parquet(GCS_PARQUET_PATH)

# 3. Load from GCS Parquet into BigQuery using native client
print("⚙ Loading from GCS Parquet into BigQuery...")
bq_client = bigquery.Client(project=PROJECT_ID)

parquet_options = bigquery.ParquetOptions()
parquet_options.enable_list_inference = True

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    parquet_options=parquet_options,
    schema=[
        bigquery.SchemaField("chunk_id", "STRING"),
        bigquery.SchemaField("repo_name", "STRING"),
        bigquery.SchemaField("chunk_idx", "INTEGER"),
        bigquery.SchemaField("chunk_text", "STRING"),
        bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),
    ]
)

load_job = bq_client.load_table_from_uri(
    f"{GCS_PARQUET_PATH}/*.parquet",
    DEST_TABLE,
    job_config=job_config
)

load_job.result()
print(f"✅ Loaded {load_job.output_rows} rows into {DEST_TABLE}")

ፁ Processing from search-me-cs226.searchme_dataset.cleaned_readme_subset to search-me-cs226.searchme_dataset.embeddings_spark_224_clean...
⚙ Writing to GCS as Parquet...
⚙ Loading from GCS Parquet into BigQuery...


/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Loaded 778 rows into search-me-cs226.searchme_dataset.embeddings_spark_224_clean


In [25]:
# --- STEP 5: FINAL ROBUST VERIFICATION ---
from pyspark.sql.functions import size, col, expr

print("ፒ Running Robust Verification...")

# 1. Load both tables
spark_v2 = spark.read.format("bigquery").option("table", f"{PROJECT_ID}.searchme_dataset.embeddings_spark_224_clean").load()
local_ref = spark.read.format("bigquery").option("table", f"{PROJECT_ID}.searchme_dataset.test_embeddings_cleaned_subset").load()

# 2. Mirror Filter: Keep only 384-dim rows from BOTH sides
clean_spark = spark_v2.filter(size(col("embedding")) == 384)
clean_local = local_ref.filter(size(col("embedding")) == 384)

# 3. Inner Join
comparison = clean_spark.alias("s").join(clean_local.alias("l"), "chunk_id")

final_count = comparison.count()
print(f"ፁ Final Validated Row Count: {final_count}")

# 4. Resilient Similarity Math
results = comparison.withColumn("similarity", expr("""
    aggregate(
        zip_with(s.embedding, l.embedding, (x, y) -> cast(x as double) * cast(y as double)),
        0.0D,
        (acc, x) -> acc + x
    )
"""))

# 5. Result Summary
avg_sim = results.selectExpr("avg(similarity)").collect()[0][0]

if avg_sim:
    print(f"ፁ MATCH ACCURACY: {avg_sim * 100:.4f}%")
    print(f"✅ Your Spark pipeline is producing the exact same results as the local run.")
else:
    print("፨ Still returning NULL. This suggests the 'embedding' column type is unexpected.")
    results.printSchema()

ፒ Running Robust Verification...
ፁ Final Validated Row Count: 778
ፁ MATCH ACCURACY: 61.6643%
✅ Your Spark pipeline is producing the exact same results as the local run.


In [26]:
%%bigquery --project search-me-cs226
SELECT
    repo_name,
    COUNT(chunk_id) as total_chunks,
    AVG(ARRAY_LENGTH(embedding)) as avg_embedding_size
FROM `searchme_dataset.embeddings_spark_224_clean`
GROUP BY 1
ORDER BY total_chunks DESC
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,repo_name,total_chunks,avg_embedding_size
0,Shinpeim/NekogataDrumSequencer,65,384.0
1,Islandora/islandora_scholar,22,384.0
2,aloisdg/Humanizer,20,384.0
3,shminer/LG-F460-Kernel,19,384.0
4,Galaxy-J5/TWRP_kernel_samsung_j5lte,19,384.0


In [27]:
# Pull a sample chunk to verify the vector values
sample_df = spark.read.format("bigquery") \
    .option("table", f"{PROJECT_ID}.searchme_dataset.embeddings_spark_224_clean") \
    .load() \
    .limit(1) \
    .toPandas()

print(f"Repo: {sample_df['repo_name'][0]}")
print(f"Chunk ID: {sample_df['chunk_id'][0]}")
print(f"First 5 vector values: {sample_df['embedding'][0][:5]}")

Repo: AveryOS/binutils
Chunk ID: AveryOS/binutils_chunk_0
First 5 vector values: [ 0.01330537 -0.05691075 -0.01107737  0.02427318  0.00378475]


In [28]:
# --- QUALITY CHECK: SPARK vs LOCAL ---
spark_run = spark.read.format("bigquery").option("table", DEST_TABLE).load()
local_run = spark.read.format("bigquery").option("table", f"{PROJECT_ID}.searchme_dataset.test_embeddings_cleaned_subset").load()

print(f"Rows in Spark table: {spark_run.count()}")
print(f"Rows in Local table: {local_run.count()}")

# Join on chunk_id and calculate similarity
comparison = spark_run.alias("s").join(local_run.alias("l"), "chunk_id")
match_count = comparison.count()
print(f"Rows matched after join: {match_count}")

if match_count > 0:
    # Debug: Check if embeddings are empty
    sample = comparison.select("chunk_id", expr("size(s.embedding)").alias("spark_vec_size"), expr("size(l.embedding)").alias("local_vec_size")).limit(3).toPandas()
    print("\n--- Vector Size Check ---")
    print(sample)

    results = comparison.withColumn("similarity", expr("""
        aggregate(
            zip_with(s.embedding, l.embedding, (x, y) -> cast(x as double) * cast(y as double)),
            0.0D,
            (acc, x) -> acc + x
        )
    """))

    # Debug: See actual similarity values
    results.select("chunk_id", "similarity").show(5)

    avg_val = results.selectExpr("avg(similarity)").collect()[0][0]
    if avg_val is not None:
        print(f"🎯 Verified Average Similarity: {avg_val:.6f}")
    else:
        print("⚠️ Similarity calculated but average is None (Check if vectors are empty or contain nulls).")
else:
    print("❌ No matching chunk_ids found between the two tables. Cannot calculate similarity.")

Rows in Spark table: 778
Rows in Local table: 1563
Rows matched after join: 778

--- Vector Size Check ---
                                  chunk_id  spark_vec_size  local_vec_size
0  Shinpeim/NekogataDrumSequencer_chunk_37             384             384
1  Shinpeim/NekogataDrumSequencer_chunk_38             384             384
2  Shinpeim/NekogataDrumSequencer_chunk_39             384             384
+--------------------+-------------------+
|            chunk_id|         similarity|
+--------------------+-------------------+
|Shinpeim/Nekogata...| 0.2239048108095535|
|Shinpeim/Nekogata...| 0.3817124261325266|
|Shinpeim/Nekogata...|0.49796021255494016|
|Shinpeim/Nekogata...|0.03215478367903677|
|Shinpeim/Nekogata...| 0.1664229755351221|
+--------------------+-------------------+
only showing top 5 rows

🎯 Verified Average Similarity: 0.616643


In [29]:
from pyspark.sql.functions import length

# --- STEP 5: PIPELINE TRACING & DEBUGGING ---
SOURCE_TABLE = f"{PROJECT_ID}.searchme_dataset.cleaned_readme_subset"

# 1. Check Source Data
df = spark.read.format("bigquery").option("table", SOURCE_TABLE).load()
print(f"Initial rows in source: {df.count()}")
df.select("readme_text", length("readme_text").alias("char_count")).show(5)

# 2. Phase A: Chunking
chunked = df.withColumn("chunks", chunk_text_udf(col("readme_text")))
print(f"Rows after chunking UDF: {chunked.count()}")

# 3. Phase B: Exploding
# Using col("chunks") as the input for explode
exploded = chunked.select("repo_name", posexplode(col("chunks")).alias("idx", "chunk_content"))
print(f"Rows after explode (individual chunks): {exploded.count()}")

# 4. Phase C: Vectorizing (Testing with small sample)
print("Generating embeddings for debug sample...")
vectored = exploded.limit(10).withColumn("embedding", generate_embeddings_udf(col("chunk_content")))
vectored.select("repo_name", "embedding").show()

Initial rows in source: 224
+----------------------+----------+
|           readme_text|char_count|
+----------------------+----------+
|  \t\t   README for...|      1719|
|  \n# Interactive M...|      1320|
|  \n### Simple Spri...|      2272|
|  \n<img src="https...|      4777|
|\nCoffeeScript 中文...|      2236|
+----------------------+----------+
only showing top 5 rows

Rows after chunking UDF: 224
Rows after explode (individual chunks): 1563
Generating embeddings for debug sample...
+--------------------+--------------------+
|           repo_name|           embedding|
+--------------------+--------------------+
|    AveryOS/binutils|[0.01292284950613...|
|    AveryOS/binutils|[-0.0403168685734...|
|    AveryOS/binutils|[0.08171326667070...|
|      unused/xenomap|[-0.0045472900383...|
|      unused/xenomap|[0.00375108071602...|
|Lex4hex/secured-shop|[-0.0356225520372...|
|Lex4hex/secured-shop|[-0.0053181019611...|
|Lex4hex/secured-shop|[-0.0155272092670...|
|        mrustl/flopy|[-0

In [30]:
test_clean = df.withColumn("cleaned", clean_markdown_udf(col("readme_text")))
test_clean.select("cleaned").show(3, truncate=200)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                                                                                 cleaned|
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|README for GNU development tools\n\nThis directory contains various GNU compilers, assemblers, linkers, \ndebuggers, etc., plus their support routines, definitions, and documentation.\n\nIf you are...|
|Interactive Map for Xenoblade Chronicles X\n\nIs an interactive map in first place to find location of enemies while troopmissions, where time is limited. Visit to see current version.\n\